In [ ]:
import os
import numpy as np
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
import math

# inisialisasi path folder dan model
MODEL_PATH = "D:/skripsi/wastecategorized13.tflite"
TEST_DIR = "D:/skripsi/test"
OUTPUT_DIR = "D:/skripsi/hasil_prediksi"
IMG_SIZE = 224

os.makedirs(OUTPUT_DIR, exist_ok=True)


# urutan folder testing
CLASS_NAMES = ["Non-daur ulang", "organik", "plastik dan kertas"]

# Load model
interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

def preprocess_image(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=0)
    return img

for class_name in CLASS_NAMES:
    class_folder = os.path.join(TEST_DIR, class_name)

    # urutkan file dalam folder
    files = sorted(os.listdir(class_folder))

    for file in files:
        img_path = os.path.join(class_folder, file)

        input_data = preprocess_image(img_path)

        interpreter.set_tensor(input_details[0]['index'], input_data)
        interpreter.invoke()

        output_data = interpreter.get_tensor(output_details[0]['index'])
        pred_index = np.argmax(output_data)
        pred_class = CLASS_NAMES[pred_index]

        img = cv2.imread(img_path)

    
        color = (0, 255, 0) if pred_class == class_name else (0, 0, 255)

       
        # kode untuk membuat header di area text 
        
        header_height = 80
        header = np.zeros((header_height, img.shape[1], 3), dtype=np.uint8)

        # Gabungkan header + gambar
        img_with_header = np.vstack((header, img))

       
        cv2.putText(img_with_header,
                    f"Actual   : {class_name}",
                    (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, color, 2)

        cv2.putText(img_with_header,
                    f"Predicted: {pred_class}",
                    (10, 65),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, color, 2)

       
        output_name = f"{os.path.splitext(file)[0]}_{class_name}_{pred_class}.png"
        output_path = os.path.join(OUTPUT_DIR, output_name)

        cv2.imwrite(output_path, img_with_header)

print("Selesai. Semua hasil sudah tersimpan sesuai urutan folder.")
